# 14장 · 한 문단으로 첫 LCEL 체인 만들기

수강생이 묻습니다. “녹음 중에 빨간불을 끌 수 있나요?” 13장의 챗봇은 질문을 보낼 수 있습니다. 하지만 제품마다 조작 방법이 다릅니다. 오늘은 사람이 먼저 정확한 설명 한 문단을 골라 주고, 모델에게 그 문단으로 답하게 합니다. 문단을 자동으로 찾아 넣는 일은 뒤 장에서 맡깁니다.

## 실습 준비

전체 실습 묶음을 풀고 노트북을 엽니다. 같은 폴더에서 시작해 자료 묶음의 `pre-mid/documents`를 읽습니다. 공용 준비 파일에 뒤에서 사용할 함수도 있으므로 **아래 셀에서 가져오는 이름**부터 살펴보세요. 처음 한 번 커널 셀에서 `%pip install langchain-openai langchain-core python-dotenv`를 실행한 뒤 커널을 재시작합니다. 8장에서 쓴 OPENAI_API_KEY를 같은 작업 폴더의 .env에 두세요. 모델을 바꾸려면 OPENAI_MODEL을 지정합니다. 호출은 계정의 사용량에 반영됩니다.

In [1]:
from pathlib import Path
import sys

# 압축을 푼 폴더 또는 그 아래에서 실습 자료의 위치를 찾습니다.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pre-mid/documents/rag/rag_lab.py").is_file()), None)
if root is None:
    raise FileNotFoundError("전체 실습 묶음을 풀고 pre-mid 폴더 옆에서 실행하세요.")
rag_dir = root / "pre-mid/documents/rag"
sys.path.insert(0, str(rag_dir))

## 질문을 받기 전에 원문으로 답해 보기

A형 설명서에는 “녹음 중에는 앞면의 빨간 표시등이 켜집니다. 녹음하는 동안 표시등을 끄는 설정은 없습니다.”라고 쓰여 있습니다. ‘표시등을 끄는 방법’에는 이 문장만으로 답할 수 있습니다. ‘배터리는 몇 시간 가나요?’에 답할 문장은 없습니다. 두 질문의 기대 행동을 먼저 적어 두세요.

**LCEL**은 LangChain 안에서 입력을 받아 결과를 내는 단계들을 연결하는 표현 방식입니다. 프롬프트는 빈칸을 채워 모델에 보낼 메시지를 만들고, 모델은 메시지에 답하고, 출력 파서는 답변에서 글을 꺼냅니다. 각 단계에 입력 하나를 주고 결과를 받을 때 쓰는 메서드가 `invoke`입니다.

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rag_lab import chat_model

manual = "A형은 녹음 중 빨간 표시등이 켜지며, 녹음하는 동안 표시등을 끄는 설정은 없습니다."
question = "A형으로 녹음하면서 표시등만 끌 수 있나요?"
prompt = ChatPromptTemplate.from_messages([
    ("system", "주어진 원문만 사용하세요. 확인할 수 없으면 어느 정보가 없는지 쓰세요."),
    ("human", "원문: {source}\n질문: {question}")])
inputs = {"source": manual, "question": question}
# 원문과 질문이 각각 어느 자리에 들어갔는지 먼저 확인합니다.
ready = prompt.invoke(inputs)
print(ready.to_messages()[-1].content)

원문: A형은 녹음 중 빨간 표시등이 켜지며, 녹음하는 동안 표시등을 끄는 설정은 없습니다.
질문: A형으로 녹음하면서 표시등만 끌 수 있나요?


질문에 표시등이 들어가도 **원문** 칸이 비었다면 모델은 근거를 받지 못합니다. 완성된 메시지를 읽으면 답이 잘못됐을 때 원문을 전달한 곳부터 확인할 수 있습니다. 같은 양식이므로 매번 긴 안내문을 다시 쓸 필요가 없습니다.

In [3]:
model = chat_model()
reply = model.invoke(ready)  # 모델은 글과 호출 정보를 함께 돌려줍니다.
parser = StrOutputParser()
print(parser.invoke(reply))

A형은 녹음 중 빨간 표시등이 켜지며, 녹음하는 동안 표시등을 끄는 설정은 없습니다.  
따라서 A형으로 녹음하면서 표시등만 끌 수는 없습니다.


### 한 번의 요청에서 사용량 읽기

답변을 글로 꺼내기 전의 reply에는 모델이 응답과 함께 돌려준 정보가 남습니다. 입력 토큰은 지시·원문·질문을 보내는 데, 출력 토큰은 답을 만드는 데 사용됩니다. 같은 모델·같은 입력으로 호출해도 실제로 생성한 문장의 길이는 달라질 수 있습니다.

In [4]:
print(reply.response_metadata.get("model_name"))
print(reply.usage_metadata)

gpt-4.1-mini-2025-04-14
{'input_tokens': 86, 'output_tokens': 52, 'total_tokens': 138, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


## 한 번의 호출로 묶기

앞 단계의 결과가 다음 단계의 입력으로 그대로 들어갑니다. Python의 `|`가 여기서는 단계를 연결합니다. 입력 딕셔너리의 키 이름은 프롬프트의 두 빈칸과 같아야 합니다. 필드 이름을 틀리면 원문 내용이 아무리 정확해도 모델에 도달하지 못합니다.

In [5]:
chain = prompt | model | parser
# 모델 호출은 새로 이루어져 같은 사실도 다른 문장으로 표현될 수 있습니다.
print(chain.invoke(inputs))

A형은 녹음 중 빨간 표시등이 켜지며, 녹음하는 동안 표시등을 끄는 설정은 없습니다.  
따라서 A형으로 녹음하면서 표시등만 끌 수는 없습니다.


## 질문을 바꿨더니 빈칸 이름 오류가 났다면

실제 앱에서는 `source`를 보낼 자리에 `document`라고 쓸 수 있습니다. 모델이 엉뚱한 답을 했을 때와 달리, 이때는 모델을 호출하기 전에 프롬프트를 만들 수 없습니다. 수업용으로 실패를 한 번 관찰합니다. 아래 셀은 오류의 종류만 표시하므로 다음 셀도 계속 실행할 수 있습니다.

In [6]:
try:
    prompt.invoke({"document": manual, "question": question})
except KeyError:
    print("프롬프트가 기다리는 입력 이름", prompt.input_variables)

프롬프트가 기다리는 입력 이름 ['question', 'source']


출력에서 source를 찾으세요. 이름을 복구하고 같은 질문에 빈 원문을 주어 보세요. 이번에는 셀이 실행돼 답이 도착해도 모델이 받았어야 할 정보가 빠졌습니다. **실행할 수 없음**과 **실행은 됐지만 근거 부족**을 구별할 수 있어야 실제 앱에서 고칠 위치가 보입니다.

15장의 PDF 읽기 전에 지금 사용하는 `source`는 사람이 직접 골랐다는 점을 기억하세요. 제품명을 빼고 같은 질문을 보냈다가 모델이 특정 기종을 추측하면 그 기종이 적힌 위치를 찾습니다. 자료가 주는 사실과 사용자가 알려 준 상황도 서로 다릅니다. [LCEL 단계 연결 참고](https://reference.langchain.com/python/langchain-core/runnables/base/RunnableSequence)

## 실습 · 같은 원문이 대답해 주는 범위 찾기

A형의 표시등 질문을 “표시등을 끄면 배터리가 몇 시간 더 가나요?”로 바꾸세요. 원문에서 알 수 있는 사실과 확인할 수 없는 수치를 구분해 답하는지 보세요. 이어 B형 설명서 1쪽의 ‘녹음 전 숨김’ 문단을 복사해 원문과 기종만 교체합니다. 답을 바꾼 정보가 질문과 원문 중 어디에 있었는지 표시하세요.

In [7]:
# inputs를 복사해 새 질문을 보내 보세요.

<details class="notebook-answer"><summary>모범 풀이와 판정 기준</summary><pre><code>limited = {**inputs, &quot;question&quot;: &quot;표시등을 끄면 배터리가 몇 시간 더 가나요?&quot;}
print(chain.invoke(limited))
other = {&quot;question&quot;: &quot;B형은 언제 표시등을 숨겨야 하나요?&quot;,
         &quot;source&quot;: &quot;B형은 녹음 전에 설정에서 표시등을 ‘숨김’으로 바꿉니다.&quot;}
print(chain.invoke(other))</code></pre><p>첫 질문에 특정 시간 수치를 단정하면 원문을 벗어났습니다. 두 번째 답은 녹음을 시작하기 전이라는 조건을 살려야 합니다. 모델 답변의 표현은 달라도 됩니다.</p><p>집필 시 실행 결과</p><pre>원문에는 표시등을 끄는 설정이 없다고 되어 있어, 표시등을 끄면 배터리가 몇 시간 더 가는지에 대한 정보가 없습니다.
B형은 녹음 전에 표시등을 ‘숨김’으로 바꿔야 합니다.</pre></details>

## 독립 실습 · 모델에게 원문을 보내지 않은 경우

A형 1.1판 1쪽의 표시등 설명으로 채팅형 프롬프트를 만드세요. 처음에는 자료 칸에 빈 문자열을, 두 번째에는 원문을 보냅니다. 학생 두 사람이 답만 읽고 어떤 실행에서 문서를 받았는지 맞혀 봅니다. 다음에는 완성 프롬프트를 열어 실제로 들어간 값을 확인하세요. 모델이 빈 원문에서도 우연히 사실과 같은 답을 했다면 근거를 전달한 것으로 인정할 수 있을까요?

| 실제로 넣은 원문 | 모델에 전달된 마지막 메시지 | 실제 답 | 원문으로 확인한 사실 |
|---|---|---|---|
| 빈 문자열 | | | |
| A형 표시등 문단 | | | |

<details class="notebook-answer"><summary>판정 예시와 해설</summary><p>원문을 비웠는데 모델이 “A형은 끌 수 없다”고 답할 수도 있습니다. 그 답은 PDF를 읽고 만든 결과인지 알 수 없습니다. 원문을 넣은 실행의 마지막 메시지에서 ‘녹음하는 동안 표시등을 끄는 설정은 없습니다’를 직접 찾으면 모델이 받은 근거를 확인할 수 있습니다. 답만 보고 문서 전달 성공이라고 표시하면 안 됩니다.</p></details>

## 다음 수업에 가져갈 것

원문·질문·실제 답을 한 묶음으로 저장하세요. 이번에는 사람이 골라 준 문장 하나로 답했습니다. 다음 장에서는 이 문장이 PDF 어디에 있는지 다시 열 수 있게 파일명과 쪽수를 함께 가져옵니다.